In [ ]:
# Standard library imports
import glob
import logging
import os
import random
import time

# Third-party imports
import boto3
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from PIL import Image

# SageMaker imports
import sagemaker
from sagemaker import image_uris, get_execution_role
from sagemaker.session import TrainingInput
from sagemaker.tuner import (
    ContinuousParameter,
    CategoricalParameter,
    HyperparameterTuner,
)

# Suppress SageMaker estimator warnings
logging.getLogger("sagemaker.estimator").setLevel(logging.ERROR)

In [ ]:
def get_random_image(dir, condition):
    placeholder = ''
    if condition == 'n':
        placeholder = 'NORMAL'
    elif condition == 'p':
        placeholder = 'PNEUMONIA'
    else:
        raise Exception('Invalid Condition.')
    folder = f'./chest_xray/{dir}/{placeholder}/*.jpeg'
    img_paths = glob.glob(folder)
    max_length = len(img_paths)
    random_number = random.randint(0, max_length)
    for index, item in enumerate(img_paths, start = 1):
        if index == random_number:
            print(index, item)
            image = plt.imread(item)
            ready_image = plt.imshow(image)
            return ready_image

In [ ]:
get_random_image('test', 'n')

In [ ]:
image = Image.open('chest_xray/test/PNEUMONIA/person109_bacteria_517.jpeg')
print(image.format)
print(image.size)
print(image.mode)

In [ ]:
# import PIL.Image
# rgba_image = PIL.Image.open('')
# rgb_image = rgba_image.convert('RGB')

In [ ]:
folder = f'./chest_xray/train/*/*.jpeg'

counter_p = 0
counter_n = 0

img_paths = glob.glob(folder)

for i in img_paths:
    if 'person' in i:
        full_size_image = Image.open(i)
        img = full_size_image.resize((224, 224))
        plt.imsave(fname='./chest_xray/train' + '/train_pneumonia_' + str(counter_p) + '.jpeg',
                   arr=img, format='jpeg', cmap='gray')
        counter_p += 1
    else:
        full_size_image = Image.open(i)
        img = full_size_image.resize((224, 224))
        plt.imsave(fname='./chest_xray/train' + '/train_normal_' + str(counter_n) + '.jpeg',
                   arr=img, format='jpeg', cmap='gray')
        counter_n += 1

In [ ]:
folder = f'./chest_xray/test/*/*.jpeg'

counter_p = 0
counter_n = 0

img_paths = glob.glob(folder)

for i in img_paths:
    if 'person' in i:
        full_size_image = Image.open(i)
        img = full_size_image.resize((224, 224))
        plt.imsave(fname='./chest_xray/test' + '/test_pneumonia_' + str(counter_p) + '.jpeg',
                   arr=img, format='jpeg', cmap='gray')
        counter_p += 1
    else:
        full_size_image = Image.open(i)
        img = full_size_image.resize((224, 224))
        plt.imsave(fname='./chest_xray/test' + '/test_normal_' + str(counter_n) + '.jpeg',
                   arr=img, format='jpeg', cmap='gray')
        counter_n += 1

In [ ]:
folder = f'./chest_xray/val/*/*.jpeg'

counter_p = 0
counter_n = 0

img_paths = glob.glob(folder)

for i in img_paths:
    if 'person' in i:
        full_size_image = Image.open(i)
        img = full_size_image.resize((224, 224))
        plt.imsave(fname='./chest_xray/val' + '/val_pneumonia_' + str(counter_p) + '.jpeg',
                   arr=img, format='jpeg', cmap='gray')
        counter_p += 1
    else:
        full_size_image = Image.open(i)
        img = full_size_image.resize((224, 224))
        plt.imsave(fname='./chest_xray/val' + '/val_normal_' + str(counter_n) + '.jpeg',
                   arr=img, format='jpeg', cmap='gray')
        counter_n += 1

In [ ]:
image = Image.open('chest_xray/train/train_normal_3.jpeg')
print(image.format)
print(image.size)
print(image.mode)

In [ ]:
folder = f'./chest_xray/*/*.jpeg'

category = []
filename = []
condition_of_lung = []

all_files = glob.glob(folder)
all_files[0]

In [ ]:
all_files[-1]

In [ ]:
for file in all_files:
    if 'train' in file:
        if 'pneumonia' in file:
            category.append('train')
            filename.append(file)
            condition_of_lung.append('pneumonia')
        elif 'normal' in file:
            category.append('train')
            filename.append(file)
            condition_of_lung.append('normal')
    elif 'test' in file:
        if 'pneumonia' in file:
            category.append('test')
            filename.append(file)
            condition_of_lung.append('pneumonia')
        elif 'normal' in file:
            category.append('test')
            filename.append(file)
            condition_of_lung.append('normal')
    elif 'val' in file:
        if 'pneumonia' in file:
            category.append('val')
            filename.append(file)
            condition_of_lung.append('pneumonia')
        elif 'normal' in file:
            category.append('val')
            filename.append(file)
            condition_of_lung.append('normal')

In [ ]:
all_data = pd.DataFrame({'dataset_type' : category,
                         'x_ray_result' : condition_of_lung,
                         'file_name' : filename})

In [ ]:
all_data.head()

In [ ]:
g = sns.catplot(
    x = 'x_ray_result',
    hue = 'x_ray_result',
    col = 'dataset_type',
    kind = 'count',
    data = all_data,
    palette = 'hls'
)

for i in range(0, 3):
    ax = g.facet_axis(0, i)
    for p in ax.patches:
        ax.text(p.get_x() + 0.325,
        p.get_height(),
        '{0:.0f}'.format(p.get_height()),
        color = 'black',
        rotation = 'horizontal',
        size = 'large')

In [ ]:
train_folder = './chest_xray/train/*.jpeg'
train_df_lst = pd.DataFrame(columns = ['labels', 's3_path'], dtype = object)
train_imgs_path = glob.glob(train_folder)
counter = 0
class_arg = ''

for i in train_imgs_path:
    if 'pneumonia' in i:
        class_arg = 1
    else:
        class_arg = 0
    train_df_lst.loc[counter] = [class_arg, os.path.basename(i)]
    counter += 1

train_df_lst.head()

In [ ]:
test_folder = './chest_xray/test/*.jpeg'
test_df_lst = pd.DataFrame(columns = ['labels', 's3_path'], dtype = object)
test_imgs_path = glob.glob(test_folder)
counter = 0
class_arg = ''

for i in test_imgs_path:
    if 'pneumonia' in i:
        class_arg = 1
    else:
        class_arg = 0
    test_df_lst.loc[counter] = [class_arg, os.path.basename(i)]
    counter += 1

test_df_lst.head()

In [ ]:
def save_to_lst(df, prefix):
    return df[['labels', 's3_path']].to_csv(
        f"{prefix}.lst", sep='\t', index=True, header=False
    )
    
save_to_lst(train_df_lst.copy(), 'train')
save_to_lst(test_df_lst.copy(), 'test')

In [ ]:
bucket='pneumonia-ai-chest-xray'
print('bucket:{}'.format(bucket))
region='us-east-1'
print('region:{}'.format(region))
roleArn='arn:aws:s3:::pneumonia-ai-chest-xray'
print('roleArn:{}'.format(roleArn))

In [ ]:
os.environ['DEFAULT_S3_BUCKET'] = bucket

In [ ]:
!aws s3 sync ./chest_xray/train s3://${DEFAULT_S3_BUCKET}/train/

In [ ]:
!aws s3 sync ./chest_xray/test s3://${DEFAULT_S3_BUCKET}/test/

In [ ]:
boto3.Session().resource('s3').Bucket(bucket).Object('train.lst').upload_file('./train.lst')

In [ ]:
boto3.Session().resource('s3').Bucket(bucket).Object('test.lst').upload_file('./test.lst')

In [ ]:
sess = sagemaker.Session()

algorithm_image = image_uris.retrieve(
    region=boto3.Session().region_name,
    framework='image-classification'
)

s3_output_location = f's3://{bucket}/models/image_model'
print(algorithm_image)

In [ ]:
role = get_execution_role()
print(role)

In [ ]:
img_classifier_model = sagemaker.estimator.Estimator(
    algorithm_image,
    role=role,
    instance_count=1,
    instance_type='ml.g4dn.xlarge',
    volume_size=50,
    max_run=432000,
    input_mode='File',
    output_path=s3_output_location,
    sagemaker_session=sess
)
print(img_classifier_model)

In [ ]:
count = 0

for filepath in glob.glob('./chest_xray/train/*.jpeg'):
    count += 1
print(count)  # 5216

In [ ]:
count = 5216

In [ ]:
img_classifier_model.set_hyperparameters(
    image_shape='3,224,224',
    num_classes=2,
    use_pretrained_model=1,
    num_training_samples=count,
    augmentation_type='crop_color_transform',
    epochs=15,  # epochs=50,
    early_stopping=True,
    early_stopping_min_epochs=8,  # early_stopping_min_epochs=30,
    early_stopping_tolerance=0.0,
    early_stopping_patience=5,
    lr_scheduler_factor=0.1,
    lr_scheduler_step='8,10,12')  # lr_scheduler_step='25,30,35'

In [ ]:
hyperparameter_ranges = {
    'learning_rate': ContinuousParameter(0.01, 0.1),
    'mini_batch_size': CategoricalParameter([8, 16, 32]),
    'optimizer': CategoricalParameter(['sgd', 'adam'])
}

In [ ]:
objective_metric_name = 'validation:accuracy'
objective_type = 'Maximize'
max_jobs = 5
max_parallel_jobs = 1

In [ ]:
tuner = HyperparameterTuner(
    estimator=img_classifier_model,
    objective_metric_name=objective_metric_name,
    hyperparameter_ranges=hyperparameter_ranges,
    objective_type=objective_type,
    max_jobs=max_jobs,
    max_parallel_jobs=max_parallel_jobs)

In [ ]:
model_inputs = {
    'train': sagemaker.inputs.TrainingInput(s3_data=f's3://{bucket}/train/',
                                            content_type='application/x-image'),
    'validation': sagemaker.inputs.TrainingInput(s3_data=f's3://{bucket}/test/',
                                                 content_type='application/x-image'),
    'train_lst': sagemaker.inputs.TrainingInput(s3_data=f's3://{bucket}/train.lst',
                                                content_type='application/x-image'),
    'validation_lst': sagemaker.inputs.TrainingInput(s3_data=f's3://{bucket}/test.lst',
                                                     content_type='application/x-image'),
}

In [ ]:
job_name_prefix = 'classifier'
timestamp = time.strftime('-%Y-%m-%d-%H-%M-%S',
                          time.gmtime())
job_name = job_name_prefix + timestamp

In [ ]:
tuner.fit(inputs=model_inputs, job_name=job_name, logs=True)

In [ ]:
role = get_execution_role()

In [ ]:
model = sagemaker.model.Model(
    image_uri=algorithm_image,
    model_data='s3://pneumonia-ai-chest-xray/models/image_model/classifier-2025-06-20-20-03-56-002-ce10222c/output/model.tar.gz',
    role=role)

In [ ]:
endpoint_name = 'pneumonia-detection-classifier'

deployment = model.deploy(
    initial_instance_count=1,
    instance_type='ml.g4dn.xlarge',
    endpoint_name=endpoint_name)

In [ ]:
from sagemaker.predictor import Predictor

predictor = Predictor('pneumonia-detection-classifier')

In [ ]:
from sagemaker.serializers import IdentitySerializer
import base64

file_name = 'chest_xray/val/val_pneumonia_0.jpeg'

predictor.serializer = IdentitySerializer('image/jpeg')
with open(file_name, 'rb') as f:
    payload = f.read()

inference = predictor.predict(data=payload)
print(inference)  # 0.6016275882720947 => pneumonia

In [ ]:
from sagemaker.serializers import IdentitySerializer
import base64

file_name = 'chest_xray/val/val_normal_0.jpeg'

predictor.serializer = IdentitySerializer('image/jpeg')
with open(file_name, 'rb') as f:
    payload = f.read()

inference = predictor.predict(data=payload)
print(inference)  # 0.43459489941596985 => normal

In [ ]:
import json
import numpy as np

file_path = 'chest_xray/val/*.jpeg'
files = glob.glob(file_path)

y_true = []
y_pred = []

In [ ]:
def make_pred():
    for file in files:
        if 'normal' in file:
            with open(file, 'rb') as f:
                payload = f.read()
                inference = predictor.predict(data=payload).decode("utf-8")
                result = json.loads(inference)
                predicted_class = np.argmax(result)
                y_true.append(0)
                y_pred.append(predicted_class)
        elif 'pneumonia' in file:
            with open(file, 'rb') as f:
                payload = f.read()
                inference = predictor.predict(data=payload).decode("utf-8")
                result = json.loads(inference)
                predicted_class = np.argmax(result)
                y_true.append(1)
                y_pred.append(predicted_class)

In [ ]:
make_pred()

In [ ]:
print(y_true)
print(y_pred)

In [ ]:
from sklearn.metrics import confusion_matrix

confusion_matrix(y_true, y_pred)

In [ ]:
from sklearn.metrics import classification_report

print(classification_report(y_true,y_pred))